# **El Objetivo de este archivo es tener una comparación rápida de los resultados del Modelo de Originación de Estructurados**
Para esto se requiere información de:
- DF_Moras_Estados: Comisiones, Pagos y Fechas de Originacion
- Cartera_Berex: Número de Plazos de Pago
- Aprobaciones: % de Primer Pago, Resultado de la Calculadora, Apartado Mensual
- Cambios de Referencia si es necesario
- Cobros Parciales

Tiempo estimado de Ejecución:

# Paso 1: **Obtener y Limpiar las Bases de Datos**

## 1.0 **Imports Generales**

In [1]:
import pandas as pd
import gspread
from datetime import datetime
from zoneinfo import ZoneInfo
import json
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.oauth2.service_account import Credentials
from gspread.exceptions import APIError
import numpy as np
from time import sleep
import os
import requests
import random
from requests.exceptions import ChunkedEncodingError, ConnectionError, Timeout
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from io import StringIO
# Importamos el diccionario con valores predefinidos
from googleapiclient.discovery import build
from collections import defaultdict
import re
import io

# Funcion Auxiliar para imputar nans sin errores
def imputeNansNoErrors(df: pd.DataFrame, col: str, value):
  maskNans = df[col].isna()
  df.loc[maskNans, col] = value

# Funcion Auxiliar para obtener variables de entorno/ secretos de forma robusta
def getEnvVar(var: str):
  try: # Si el entorno es Colab las credenciales se obtendrán de esa forma
    from google.colab import userdata
    return json.loads(userdata.get(var))
  except ImportError: # Si el entorno es GitHub las credenciales se obtienen es mediante variables de entorno
    creds = os.getenv(var)
    if not creds:
      raise ValueError("No se encontraron las credenciales en las variables de entorno.")
    return json.loads(creds.strip())

# Función Auxiliar para obtener credenciales
def getCreds():
  try: # Si el entorno es Colab las credenciales se obtendrán de esa forma
    from google.colab import userdata
    return json.loads(userdata.get('MI_JSON'))
  except ImportError: # Si el entorno es GitHub las credenciales se obtienen es mediante variables de entorno
    creds = os.getenv('MI_JSON')
    if not creds:
      raise ValueError("No se encontraron las credenciales en las variables de entorno.")
    return json.loads(creds.strip())

# 1. Retrieve the secret from Colab
service_account_info = getCreds()

# 2. Define the scope and authenticate
scope = ['https://www.googleapis.com/auth/spreadsheets',
         'https://www.googleapis.com/auth/drive']

creds = Credentials.from_service_account_info(service_account_info, scopes=scope)
client = gspread.authorize(creds)

## 1.1 **Datos de DF_MORAS**
Datos que se obtienen: **morasDf**
- Referencia
- Fecha_Origen
- Cobro_Actual
- Pago_Actual
- Porcentaje_Pago

In [2]:
#### Traemos los Datos desde Sheets
# Abrimos la SpreadSheet
morasShId = '1jcPPhtF2YK3Kr7P_A0Mgh2OqhOfnVWB2to3UPoSH5tE'
morasSH = client.open_by_key(morasShId)
# Abrimos la WorkSheet
morasWsName = 'Comisión'
morasWs = morasSH.worksheet(morasWsName)
# Obtenemos los datos
morasDf = get_as_dataframe(morasWs, evaluate_formulas=True)

# Dejamos las columnas Requeridas
moraCols = ['REFERENCIA','FECHA','FECHA_ORIGEN','X_COBRAR','PAGO']
morasDf = morasDf[moraCols]

# Renombrar las Columnas a nombres más amigables
morasDf = morasDf.rename(columns={'REFERENCIA':'Referencia','FECHA':'Fecha','FECHA_ORIGEN':'Fecha_Origen','X_COBRAR':'Por_Cobrar','PAGO':'Pago'})

# Volvemos la Columna Referencia a string
morasDf['Referencia'] = morasDf['Referencia'].apply(lambda x: str(x).replace('.0','').strip())

# Volvemos la Columna Fecha_Origen y Fecha a Datetime
morasDf['Fecha_Origen'] = pd.to_datetime(morasDf['Fecha_Origen'], errors='coerce', dayfirst=False)
morasDf['Fecha'] = pd.to_datetime(morasDf['Fecha'], errors='coerce', dayfirst=False)

### Ahora la idea es asignar la Fecha_Origen correcta a todos los requeridos
# Se realiza ordenando por Referencia y Fecha
# Así se guardan los índices hasta que se encuentre la siguiente fecha de originacion
# Mientras no exista una fecha de originación se van agregando los índices
# Cuando aparezca una fecha de originación nueva los indices se actualizan por la fecha de originación

# 1. Ordenar por Referencia y Fecha
morasDf = morasDf.sort_values(by=['Referencia', 'Fecha'])

# 2. Iterar por cada una de las filas
# Definimos la currFechaOrigen como nula, la referencia actual como nula y los indices como vacios
currFechaOrigen = None
indices = []
for idx, row in morasDf.iterrows():
    # 3. Verificar si es existe una fecha de origen (no es NaN)
    if not pd.isna(row['Fecha_Origen']):
      # 4. Actualizar los índices si hay
      if indices:
        morasDf.loc[indices, 'Fecha_Origen'] = currFechaOrigen
      # 5. Actualizar la fecha de origen que se lleva hasta el momento
      currFechaOrigen = row['Fecha_Origen']
      currRef = row['Referencia']
      # Reiniciamos los Indices
      indices = [idx]
      # Se continua con la ejecucion
      continue
    # 6. Si no existe una fecha de origen se añade el indice a la lista
    indices.append(idx)
if indices:
  # Se realiza la actualizacion final
  morasDf.loc[indices, 'Fecha_Origen'] = currFechaOrigen

# Ahora se va a filtrar por fechas de origen mayor o igual al la fecha inicial y menor a la fecha final destinadas
startDate = pd.Timestamp(year=2025,month=11,day=19) # Desde este día hay datos de la calculadora
# La fecha final será el último día del mes anterior a hoy
# Se hace definiendo el primer dia del mes y quitandole un dia
# Tambien se va a dejar el dia a las 23:59:59 para dejar todos los cientes que asegurados fue en el mes anterior
today = datetime.now()
endDate = today.replace(day=1, hour=23, minute=59, second=59, microsecond=0)
endDate = endDate - pd.Timedelta(days=1)

# Filtramos por fecha
morasDf = morasDf[(morasDf['Fecha_Origen'] >= startDate) & (morasDf['Fecha_Origen'] <= endDate)]

# Ahora se vuelven las columnas Por_Cobrar y Pago a numerico
morasDf['Por_Cobrar'] = pd.to_numeric(morasDf['Por_Cobrar'], errors='coerce')
morasDf['Pago'] = pd.to_numeric(morasDf['Pago'], errors='coerce')

# Ahora se imputan los Nans de Por_Cobrar y Pago a 0
imputeNansNoErrors(morasDf, 'Por_Cobrar', 0)
imputeNansNoErrors(morasDf, 'Pago', 0)

### Ahora se va a realizar la agrupación por Referencia y Fecha de Origen
# La agrupación va a agregar la suma total de Por_Cobrar y la Suma total de Pago
morasDf = morasDf.groupby(['Referencia', 'Fecha_Origen']).agg({'Por_Cobrar': 'sum', 'Pago': 'sum'}).reset_index()

# Renombramos Columnas Por_Cobrar a Cobro_Actual y Pago a Pago_Actual
morasDf = morasDf.rename(columns={'Por_Cobrar': 'Cobro_Actual', 'Pago': 'Pago_Actual'})

# Se imputan Nans de Cobro_Actual y Pago_Actual a 0
imputeNansNoErrors(morasDf, 'Cobro_Actual', 0)
imputeNansNoErrors(morasDf, 'Pago_Actual',0)

### Se crea una nueva columna llamada Porcentaje_Pago, que sera la división entre el cobro_Actual y el Pago_Actual
# Como el cobro_Actual puede ser 0, hay que crear una función para manejar este caso
def calcPorcentajePago(row):
  if row['Cobro_Actual'] == 0:
    return 0
  # Se deja 1 como maximo para realizar una buena comparación
  return min(row['Pago_Actual'] / row['Cobro_Actual'], 1)
# Añadimos la Columna redondeando a 5 decimales
morasDf['Porcentaje_Pago'] = morasDf.apply(calcPorcentajePago, axis=1).round(5)

# Por Último convertimos la columna Fecha_Origen a string en formato: %Y-%m-%d
morasDf['Fecha_Origen'] = morasDf['Fecha_Origen'].dt.strftime('%Y-%m-%d')

print('✅Moras cargadas con éxito, columnas: {}'.format(', '.join(morasDf.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(morasDf)))

✅Moras cargadas con éxito, columnas: Referencia, Fecha_Origen, Cobro_Actual, Pago_Actual, Porcentaje_Pago
🚼Filas totales: 505 filas.


## 1.2 **Datos de Cartera_Berex**
Contiene los Datos de: **carteraDf**
- Referencia
- Fecha_Origen
- Primer_PaB
- Plazos_Estructurado
- Total_Pago_Estruct

In [24]:
#### Traemos los Datos desde Sheets
# Abrimos la SpreadSheet
carteraShId = '13Vf32LzRI2V95dIUqfevzm-ZmsDR3d17UTre_7XJ-UU'
carteraSH = client.open_by_key(carteraShId)
# Abrimos la Worksheet
carteraWsName = 'Cartera'
carteraWs = carteraSH.worksheet(carteraWsName)
# Obtenemos los datos
carteraDf = get_as_dataframe(carteraWs, evaluate_formulas=True)

# Dejamos las columnas Requeridas
carteraCols = ['reference','Originado','amount','destination','payment_date']
carteraDf = carteraDf[carteraCols]

# Renombramos las Columnas a Nombres más entendibles
carteraDf = carteraDf.rename(columns={'reference':'Referencia','Originado':'Fecha_Origen',
                                      'amount':'Total_Pago_Estruct'})

# Volvemos la Columna Referencia a string
carteraDf['Referencia'] = carteraDf['Referencia'].apply(lambda x: str(x).replace('.0','').strip())

# Convertimos la Columna Fecha_Origen a Datetime
carteraDf['Fecha_Origen'] = pd.to_datetime(carteraDf['Fecha_Origen'], errors='coerce', dayfirst=True)

# Dejamos las Fecha_Origens que esten entre el rango propuesto en la carga de datos de DfMoras
carteraDf = carteraDf[(carteraDf['Fecha_Origen'] >= startDate) & (carteraDf['Fecha_Origen'] <= endDate)]

# Creamos la Columna Primer_PaB con logica:
# pd.NA if destination != 'bank' else Total_Pago_Estruct
carteraDf['Primer_PaB'] = np.where(carteraDf['destination'] != 'bank', pd.NA, carteraDf['Total_Pago_Estruct'])

# Ordenamos por Fecha_Origen y Referencia
carteraDf = carteraDf.sort_values(by=['Fecha_Origen', 'Referencia'])

# Ahora Agrupamos por Referencia y Fecha Origen y dejamos el maximo Numero_Pago así como la suma del Pago
carteraDf = carteraDf.groupby(['Referencia', 'Fecha_Origen']).agg({'Total_Pago_Estruct':'sum','Primer_PaB':'first','payment_date':'nunique'}).reset_index()

# Renombramos la Columna payment_date a Plazos_Estructurado y Pago a Pago_Total
carteraDf = carteraDf.rename(columns={'payment_date': 'Plazos_Estructurado'})

# Por Último convertimos la columna Fecha_Origen a string en formato: %Y-%m-%d
carteraDf['Fecha_Origen'] = carteraDf['Fecha_Origen'].dt.strftime('%Y-%m-%d')

print('✅Cartera cargada con éxito, columnas: {}'.format(', '.join(carteraDf.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(carteraDf)))

✅Cartera cargada con éxito, columnas: Referencia, Fecha_Origen, Total_Pago_Estruct, Primer_PaB, Plazos_Estructurado
🚼Filas totales: 453 filas.


## 1.3 **Datos de Aprobaciones**
Datos que se obtienen: **aprobacionesDf**
- Referencia
- Fecha_Origen
- Valor_Comision_Total
- Valor_Primer_Comision
- Apartado_Mensual
- Prob_Calculadora

In [4]:
#### Traemos los Dats desde Sheets
# Abrimos la Spreadsheet
aprobacionesShId = '1Aahltn7TSRf6ZpTpS-vPgpB89hO-r5KxpAhqKAPXziE'
aprobacionesSH = client.open_by_key(aprobacionesShId)
# Abrimos la Worksheet
aprobacionesWsName = 'Respuestas Estr'
aprobacionesWs = aprobacionesSH.worksheet(aprobacionesWsName)
# Obtenemos los Datos
aprobacionesDf = get_as_dataframe(aprobacionesWs, evaluate_formulas=True)

# Dejamos las Columnas Requeridas
aprobacionesCols = ['Marca temporal','Referencia','Ingrese el valor total de comisión','Ingrese el valor del pago de la primera comisión',
                    'Aprobación Estructurados','Apartado Mensual','Calculadora']
aprobacionesDf = aprobacionesDf[aprobacionesCols]

# Renombramos las Columnas a Nombres más coherentes
aprobacionesDf = aprobacionesDf.rename(columns={'Marca temporal':'Fecha_Origen','Ingrese el valor total de comisión': 'Valor_Comision_Total',
                                                'Ingrese el valor del pago de la primera comisión': 'Valor_Primer_Comision',
                                                'Aprobación Estructurados': 'Aprobacion_Estructurados',
                                                'Apartado Mensual': 'Apartado_Mensual',
                                                'Calculadora': 'Prob_Calculadora'})

# Volvemos la Columna Fecha_Origen a Datetime
aprobacionesDf['Fecha_Origen'] = pd.to_datetime(aprobacionesDf['Fecha_Origen'], errors='coerce', dayfirst=True)

# Volvemos la Columna Referencia a string
aprobacionesDf['Referencia'] = aprobacionesDf['Referencia'].apply(lambda x: str(x).replace('.0','').strip())

# Dejamos las Fecha_Origens que esten entre el rango propuesto en la carga de datos de DfMoras
aprobacionesDf = aprobacionesDf[(aprobacionesDf['Fecha_Origen'] >= startDate) & (aprobacionesDf['Fecha_Origen'] <= endDate)]

# Volvemos las Columnas Valor_Comision_Total,Valor_Primer_Comision,Apartado_mensual y Prob_Calculadora a Números
aprobacionesDf['Valor_Comision_Total'] = pd.to_numeric(aprobacionesDf['Valor_Comision_Total'], errors='coerce')
aprobacionesDf['Valor_Primer_Comision'] = pd.to_numeric(aprobacionesDf['Valor_Primer_Comision'], errors='coerce')
aprobacionesDf['Apartado_Mensual'] = pd.to_numeric(aprobacionesDf['Apartado_Mensual'], errors='coerce')
aprobacionesDf['Prob_Calculadora'] = pd.to_numeric(aprobacionesDf['Prob_Calculadora'], errors='coerce')

# Imputamos los Nans de Valor_Comision_Total,Valor_Primer_Comision,Apartado_mensual y Prob_Calculadora a Números a 0
imputeNansNoErrors(aprobacionesDf, 'Valor_Comision_Total', 0)
imputeNansNoErrors(aprobacionesDf, 'Valor_Primer_Comision', 0)
imputeNansNoErrors(aprobacionesDf, 'Apartado_Mensual', 0)
imputeNansNoErrors(aprobacionesDf, 'Prob_Calculadora', 0.5) # Excepcion ya que un valor de 0 sería imposible por la calculadora
# Por lo que se deja por 50% (0.5)

# Redondeamos valores de Prob_Calculadora mayores a 0.9 a 1
maskProbAlta = aprobacionesDf['Prob_Calculadora'] > 0.9
aprobacionesDf.loc[maskProbAlta, 'Prob_Calculadora'] = 1

# Dejamos las columnas que hayan sido aprobadas por Estructurados
maskAprobadoEstructurados = aprobacionesDf['Aprobacion_Estructurados'].astype(bool)
aprobacionesDf = aprobacionesDf[maskAprobadoEstructurados]

# Eliminamos las Columnas de Aprobacion_Estructurados y Aprobacion_Finanzas
aprobacionesDf = aprobacionesDf.drop(columns=['Aprobacion_Estructurados'])

print('✅Aprobaciones cargada con éxito, columnas: {}'.format(', '.join(aprobacionesDf.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(aprobacionesDf)))

✅Aprobaciones cargada con éxito, columnas: Fecha_Origen, Referencia, Valor_Comision_Total, Valor_Primer_Comision, Apartado_Mensual, Prob_Calculadora
🚼Filas totales: 485 filas.


## 1.4 **Datos de Reparadora**
Datos que se obtienen: **reparadoraDf**
- Referencia
- Comision_Mensual

In [5]:
# ========= Secrets (Colab o ENV) =========
def _get_secret(name: str):
    try:
        from google.colab import userdata as _ud
        val = _ud.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

USER_API   = _get_secret("USER_API")
SECRET_API = _get_secret("SECRET_API")
if not USER_API or not SECRET_API:
    raise ValueError("Falta USER_API o SECRET_API.")

# ========= Constantes =========
BASE_URL  = "https://mutatio-api.gobravo.dev"
RESOURCE  = "/accounting/repairs/download"

# ========= Sesión con reintentos =========
def make_session():
    s = requests.Session()
    retry = Retry(
        total=3, connect=2, read=2, backoff_factor=1.2,
        status_forcelist=[502,503,504], allowed_methods={"POST"},
        raise_on_status=False
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.mount("http://",  HTTPAdapter(max_retries=retry))
    return s

session = make_session()

# ========= Auth =========
def get_token(user: str, secret: str) -> str:
    r = session.post(
        f"{BASE_URL}/auth/generate-token",
        json={"user": user, "secret": secret},
        headers={"Content-Type":"application/json","Accept":"application/json"},
        timeout=(10,45)
    )
    r.raise_for_status()
    tok = r.json().get("token")
    if not tok:
        raise RuntimeError("Auth OK pero no vino 'token'.")
    return tok

# ========= Util: leer CSV con ; o , =========
def _leer_csv_flexible(resp) -> pd.DataFrame:
    resp.raise_for_status()
    txt = resp.text or ""
    if not txt.strip():
        return pd.DataFrame()
    first = txt.splitlines()[0]
    sep = ';' if first.count(';') >= first.count(',') else ','
    return pd.read_csv(io.StringIO(txt), sep=sep, dtype=str)

# ========= Descarga =========
def descargar_pagina_repairs(user: str, secret: str, page: int, filtros=None) -> pd.DataFrame:
    if filtros is None:
        filtros = []
    url = f"{BASE_URL}{RESOURCE}"

    def _do(tok):
        return session.post(
            url,
            headers={
                "Authorization": f"Bearer {tok}",
                "Accept": "text/csv, text/plain",
                "Content-Type": "application/json",
            },
            params={"pageToDownload": str(page)},
            json=filtros,
            timeout=(12,90)
        )

    token = get_token(user, secret)
    r = _do(token)
    if r.status_code == 401:
        token = get_token(user, secret)
        r = _do(token)

    if r.status_code >= 400:
        raise RuntimeError(f"Error {r.status_code} en page={page}. {r.text[:300]}")

    return _leer_csv_flexible(r)

def descargar_todo_repairs(user: str, secret: str, max_pages: int = 1000, filtros=None) -> pd.DataFrame:
    frames = []
    for page in range(max_pages):
        df = descargar_pagina_repairs(user, secret, page=page, filtros=filtros)
        if df.empty:
            break
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# ========= Ejecutar =========
reparadoraDf = descargar_todo_repairs(USER_API, SECRET_API, max_pages=1200)

# Se dejan solo las Columnas Previstas
reparadoraCols = ['Referencia','Comision Mensual']
reparadoraDf = reparadoraDf[reparadoraCols]

# Se cambia Referencia a String
reparadoraDf['Referencia'] = reparadoraDf['Referencia'].apply(lambda x: str(x).replace('.0','').strip())

# Renombramos la Columna Comision Mensual a Comision_Mensual
reparadoraDf = reparadoraDf.rename(columns={'Comision Mensual':'Comision_Mensual'})

# Volvemos la Columna Comision_Mensual a número
reparadoraDf['Comision_Mensual'] = pd.to_numeric(reparadoraDf['Comision_Mensual'],errors='coerce')

print('✅Reparadora cargada con éxito, columnas: {}'.format(', '.join(reparadoraDf.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(reparadoraDf)))

✅Reparadora cargada con éxito, columnas: Referencia, Comision_Mensual
🚼Filas totales: 58989 filas.


## 1.5 **Datos de Cobros Parciales**
Datos que se Obtienen: **facturacionesDf**
- Referencia
- Fecha_Origen
- Status_Facturacion

In [6]:
# =======================
# Secrets (Colab / GitHub / local)
# =======================
# Solo en Colab (GHA define GITHUB_ACTIONS="true")
if 'GITHUB_ACTIONS' not in os.environ:
    try:
        from google.colab import userdata
        USER_API   = userdata.get("USER_API")
        SECRET_API = userdata.get("SECRET_API")
    except Exception:
        USER_API = SECRET_API = None
else:
    USER_API   = os.environ.get("USER_API")
    SECRET_API = os.environ.get("SECRET_API")

if not USER_API or not SECRET_API:
    raise ValueError("Falta USER_API o SECRET_API o no diste acceso desde 'Secretos'.")

# =======================
# Config
# =======================
BASE_URL = "https://mutatio-api.gobravo.dev"

def make_session():
    s = requests.Session()
    retry = Retry(
        total=5, connect=3, read=3, backoff_factor=1.5,
        status_forcelist=[500, 502, 503, 504],
        allowed_methods={"POST"},
        raise_on_status=False,
        respect_retry_after_header=True
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s

session = make_session()

# =======================
# Auth
# =======================
def get_token(user: str, secret: str) -> str:
    url = f"{BASE_URL}/auth/generate-token"
    r = session.post(
        url,
        json={"user": user, "secret": secret},
        headers={"Content-Type":"application/json","Accept":"application/json"},
        timeout=(10,45)
    )
    r.raise_for_status()
    tok = r.json().get("token")
    if not tok:
        raise RuntimeError("Auth OK pero no vino 'token'.")
    return tok

# =======================
# POST robusto con backoff
# =======================
def _post_with_backoff(url, headers, params=None, json=None,
                       max_tries=6, base_delay=1.0, read_timeout=180):
    params = params or {}
    json   = json or {}
    last_err = None
    for attempt in range(1, max_tries + 1):
        try:
            r = session.post(url, headers=headers, params=params, json=json,
                             timeout=(12, read_timeout))
            # 401: dejar que el caller regenere token y reintente
            if r.status_code == 401:
                return r
            # 429: respetar Retry-After si viene
            if r.status_code == 429:
                ra = r.headers.get("Retry-After")
                wait = float(ra) if (ra and ra.isdigit()) else base_delay*(2**(attempt-1))
                print(f"[{r.status_code}] Rate limited. Reintento en {wait:.1f}s…")
                sleep(wait); continue
            # 5xx: reintentar
            if 500 <= r.status_code < 600:
                raise requests.HTTPError(f"Server {r.status_code}", response=r)
            # 2xx / 4xx (≠429) → devolver tal cual
            return r
        except (requests.Timeout, requests.ConnectionError, requests.HTTPError) as e:
            last_err = e
            # si es 4xx ≠429, no insistir
            if isinstance(e, requests.HTTPError) and getattr(e, "response", None):
                sc = e.response.status_code
                if 400 <= sc < 500 and sc != 429:
                    break
            sleep_s = base_delay*(2**(attempt-1)) + random.uniform(0, 0.5)
            print(f"[POST RETRY {attempt}/{max_tries}] {type(e).__name__}: {e}. Reintento en {sleep_s:.1f}s…")
            sleep(sleep_s)
    if last_err:
        raise last_err
    raise requests.HTTPError("No response after retries")

# =======================
# Descarga página (facturations) robusta
# =======================
def descargar_pagina(token: str, page: int, filtros: list) -> pd.DataFrame:
    url = f"{BASE_URL}/accounting/facturations/download"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "text/csv",
        "Content-Type": "application/json",
    }
    params = {"pageToDownload": str(page)}

    # 1er intento con el token actual
    r = _post_with_backoff(url, headers, params=params, json=filtros, read_timeout=180)

    # Si fue 401, refresca token y reintenta una vez
    if r.status_code == 401:
        print("[WARN] 401: token inválido o vencido. Generando nuevo token y reintentando…")
        new_token = get_token(USER_API, SECRET_API)
        headers["Authorization"] = f"Bearer {new_token}"
        r = _post_with_backoff(url, headers, params=params, json=filtros, read_timeout=180)

    r.raise_for_status()

    if not r.content or r.content.strip() == b"":
        return pd.DataFrame()

    # Autodetecta separador ';' o ','
    first = r.content.split(b"\n", 1)[0]
    sep = b';' if first.count(b';') >= first.count(b',') else b','
    return pd.read_csv(io.BytesIO(r.content), sep=sep.decode(), dtype=str)

# =======================
# Filtro y ejecución
# =======================
filtro_liq_col = [{
    "column": "Comision",
    "attribute": "commissionType",
    "columnType": "ENUM",
    "table": "facturation",
    "operation": "IGUAL_A",
    "value": "LIQUIDACION_COLOMBIA"
}]

# Llama protegido para no tumbar el pipeline si el backend falla
try:
    token = get_token(USER_API, SECRET_API)
    facturacionesDf = descargar_pagina(token, page=0, filtros=filtro_liq_col)
except Exception as e:
    print(f"⚠️ Descarga falló: {e}. Continuo con DataFrame vacío.")
    facturacionesDf = pd.DataFrame()

# Limpieza de Dataframe de facturaciones

# Se dejan solo las Columnas Requeridas
facturacionesCols = ['Referencia','Fecha de facturacion','Status facturacion']
facturacionesDf = facturacionesDf[facturacionesCols]

# Renombramiento de las Columnas
facturacionesDf = facturacionesDf.rename(columns={'Fecha de facturacion':'Fecha_Origen','Status facturacion':'Status_Facturacion'})

# Limpiamos la Columna Referencia
facturacionesDf['Referencia'] = facturacionesDf['Referencia'].apply(lambda x: str(x).replace('.0','').strip())

# Convertimos la Columna Fecha_Origen a DateTime
facturacionesDf['Fecha_Origen'] = pd.to_datetime(facturacionesDf['Fecha_Origen'], errors='coerce', dayfirst=True)

# Dejamos solo las filas con Status_Facturacion: 'COBRO_PARCIAL_COBRADO', 'COBRO_PARCIAL'
facturacionesDf = facturacionesDf[facturacionesDf['Status_Facturacion'].isin(['COBRO_PARCIAL_COBRADO', 'COBRO_PARCIAL'])]

print('✅Facturaciones cargadas con éxito, columnas: {}'.format(', '.join(facturacionesDf.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(facturacionesDf)))

✅Facturaciones cargadas con éxito, columnas: Referencia, Fecha_Origen, Status_Facturacion
🚼Filas totales: 9066 filas.


## **1.6 Cambios de Referencia**

In [7]:
# Abrimos la SpreadSheet
refChangesShId = '1jcPPhtF2YK3Kr7P_A0Mgh2OqhOfnVWB2to3UPoSH5tE'
refChangesSH = client.open_by_key(refChangesShId)
# Aca se cargan los cambios de referencia
refChangesWS = refChangesSH.worksheet('Cambios de Referencia')
records = refChangesWS.get_all_values()
## La llave sera la referencia nueva y el valor la referencia vieja
refChangesDict = {str(row[1]).replace('.0','').strip():str(row[0]).replace('.0','').strip() for row in records}

print('✅Cambios de Referencia Cargadas Correctamente!')

✅Cambios de Referencia Cargadas Correctamente!


# Paso 2: **Unión de los Datos**
La union sigue este eschema:
- Union 1: moras y cartera (inner)
- Union 2: Union 1 y aprobaciones (left por si hay algun error)

### **Lógica de la Unión**
Puede que hayan fechas de originación con días de diferencia, por esta razón vamos a realizar una unión que permita una diferencia máxima de 2 días

In [25]:
# Actualizamos las Referencias de cada Df
# Referencias de morasDf
morasDf['Referencia'] = morasDf['Referencia'].apply(lambda x: refChangesDict.get(x, x))
# Referencias de Cartera
carteraDf['Referencia'] = carteraDf['Referencia'].apply(lambda x: refChangesDict.get(x, x))
# Referencias de Aprobaciones
aprobacionesDf['Referencia'] = aprobacionesDf['Referencia'].apply(lambda x: refChangesDict.get(x, x))
# Referencias de Reparadora
reparadoraDf['Referencia'] = reparadoraDf['Referencia'].apply(lambda x: refChangesDict.get(x, x))
# Referencias de Facturaciones
facturacionesDf['Referencia'] = facturacionesDf['Referencia'].apply(lambda x: refChangesDict.get(x, x))

In [26]:
# Union entre Moras y Cartera
morasCarteraDf = pd.merge(morasDf, carteraDf, on=['Referencia', 'Fecha_Origen'], how='inner')

# Union entre la union anterior y reparadoraDf por Referencia
morasCarteraDf = pd.merge(morasCarteraDf, reparadoraDf, on='Referencia', how='left')

# Rellenamos los NaNs de Comision_Mensual con 0
imputeNansNoErrors(morasCarteraDf, 'Comision_Mensual', 0)

# Volvemos la Fecha_Origen DateTime de nuevo
morasCarteraDf['Fecha_Origen'] = pd.to_datetime(morasCarteraDf['Fecha_Origen'], errors='coerce', dayfirst=False)

# Ordenamos ambos Dataframes previos a unir por Fecha_Origen
morasCarteraDf = morasCarteraDf.sort_values(by='Fecha_Origen')
aprobacionesDf = aprobacionesDf.sort_values(by='Fecha_Origen')

# Union entre union anterior y Aprobaciones con diferencia de 2 dias maximo
totalDf = pd.merge_asof(morasCarteraDf, aprobacionesDf, on='Fecha_Origen',by='Referencia', direction='nearest', tolerance=pd.Timedelta(days=2))

# Ordenamos de nuevo dataframes por Fecha_Origen
facturacionesDf = facturacionesDf.sort_values(by='Fecha_Origen')
totalDf = totalDf.sort_values(by='Fecha_Origen')

# Union entre union anterior y Facturaciones con diferencia de 5 días máximo
totalDf = pd.merge_asof(totalDf, facturacionesDf, on='Fecha_Origen',by='Referencia', direction='nearest', tolerance=pd.Timedelta(days=5))

# Quitamos Datos que no esten en aprobacionesDf (Valor_Primer_Comision es NaN)
totalDf = totalDf.dropna(subset=['Valor_Primer_Comision'])

### Ahora se crea la Columna Cuota_Mensual/AM (Apartado Mensual)
# Primero se tiene que crear el calculo de la Cuota Mensual
# La cuota mensual es (Total_Pago_Estruct - Primer_PaB - Valor_Primer_Comision + (Comision Mensual * 1.19)) / Plazos_Estructurado
def calcCuotaMensual(row):
  return (row['Total_Pago_Estruct'] - row['Primer_PaB'] - row['Valor_Primer_Comision'] + (row['Comision_Mensual'] * 1.19))/ max(row['Plazos_Estructurado']-1,1)
# Se añade la columna de cuota Mensual
totalDf['Cuota_Mensual'] = totalDf.apply(calcCuotaMensual, axis=1)
# Se crea la funcion auxiliar para calcular Cuota_Mensual/AM
def calcCuotaMensualAM(row):
  if row['Apartado_Mensual'] == 0:
    return 0
  return row['Cuota_Mensual'] / row['Apartado_Mensual']

# Se crea la Columna de Cuota_Mensual/AM
totalDf['Cuota_Mensual/AM'] = totalDf.apply(calcCuotaMensualAM, axis=1)

# Creamos la Columna Tiene_Cobro_Parcial
# La Lógica es: 'Si' if Status_Facturacion notna else 'No'
totalDf['Tiene_Cobro_Parcial'] = np.where(totalDf['Status_Facturacion'].notna(), 'Si', 'No')

# Eliminamos la Columna de Status_Facturacion
totalDf = totalDf.drop(columns=['Status_Facturacion'])

# Quitamos los Datos con valores nulos en la columna Apartado_Mensual
totalDf = totalDf.dropna(subset=['Apartado_Mensual'])

print('✅Union Realizada Correctamente!, con un total de: {} datos'.format(len(totalDf)))

✅Union Realizada Correctamente!, con un total de: 393 datos


# Paso 3: **Subir los Datos a Drive**
El lugar de Subida será la misma Hoja de DF_Moras con nombre **RealVSCalc**

## 3.0 **Funciones Adicionales para la Subida de Datos**

In [27]:
# Función Auxiliar para reintentar cualquier llamda al API de sheets
def _retry(fn, label="", tries=10, base_sleep=1.5, jitter=0.6, max_sleep=45):
    RETRIABLE_CODES = ["[500]", "[502]", "[503]", "[504]", "[429]"]
    last_err = None
    for i in range(tries):
        try:
            return fn()
        except APIError as e:
            last_err = e
            msg = str(e)
            # Si el error fue del servidor
            if any(c in msg for c in RETRIABLE_CODES):
                sleep_s = min((base_sleep) * (2 ** i) + np.random.uniform(0, jitter), max_sleep)
                print(f"[RETRY {i+1}/{tries}] {label} -> {msg[:120]}... sleep {sleep_s:.1f}s")
                # Esperamos para no saturar al API
                sleep(sleep_s)
                continue
            raise
    raise last_err

# Funcion Auxiliar para obtener worksheet con objeto sheet
def getWorksheet(sh, wsName, colLen):
  try:
    return True, sh.worksheet(wsName)
  except:
    return False, sh.add_worksheet(wsName, 1000, colLen)


def uploadToSheets(ws, df, retry_label="Upload Data"):
    """
    Uploads a DataFrame to a Google Sheet, ensuring column alignment
    and cleaning up unnecessary columns.
    """
    _retry(lambda: ws.clear(), label="clear sheet")

    _retry(lambda: set_with_dataframe(ws, df), label=f"{retry_label} ({len(df)} rows)")

## 3.1 **Subida de Datos**

In [28]:
# Ordenamos las Columnas
totalOrder = ['Fecha_Origen','Referencia','Cobro_Actual','Pago_Actual','Valor_Comision_Total','Valor_Primer_Comision',
              'Apartado_Mensual','Cuota_Mensual/AM','Tiene_Cobro_Parcial','Porcentaje_Pago','Prob_Calculadora']
totalDf = totalDf[totalOrder]

# Ahora ordenamos por Referencia y Fecha_Origen
totalDf = totalDf.sort_values(by=['Fecha_Origen','Referencia'])

# Obtenemos el Spreadsheet (Estructurados Calculadora Originación)
calcSHId = '1xGSzneJkRqREZupLshG9jaNK8wbP8WpV-yjUs_Lm7AA'
calcSH = client.open_by_key(calcSHId)

# Ahora obtenemos la Worksheet
calcWsName = 'RealVSCalc'
exists, calcWs = getWorksheet(calcSH, calcWsName, len(totalDf.columns))

# Como toda la información que se trae es histórica, es más fácil subir de nuevo todo el DF a realizar la actualización
print('🚼Se esta subiendo la información')
uploadToSheets(calcWs, totalDf)
print('\n✅Datos Subidos Correctamente')

print('\n🆔En total hay actualmente {} datos.'.format(len(totalDf)))

🚼Se esta subiendo la información

✅Datos Subidos Correctamente

🆔En total hay actualmente 393 datos.
